# analysis_regime_window_diagnostics_v1

目标：把不同区间和 regime 下的模型效果差异沉到 `src/analysis` 稳定框架，notebook 只负责复现、审阅和记录中文结论。skill 只保留发散思路，不作为功能入口。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'src').is_dir() and (candidate / 'AGENTS.md').exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError('repo root not found')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis import (
    analyze_model_effects_by_regime_and_window,
    build_model_regime_window_diagnostics,
    build_regime_window_analysis_report,
    compare_models,
    compute_regime_metrics,
    compute_window_metrics,
    write_analysis_report,
)
from src.core import DatasetSnapshot

CARD_DIR = REPO_ROOT / 'research/analysis_regime_window_diagnostics_v1'
REPORT_DIR = CARD_DIR
CARD_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


## 输入样本

这里使用最小可复现的 daily backtest 风格表。真实 qlib recorder 结果会先通过 `load_qlib_recorder_analysis` 标准化成相同字段，再进入同一套分析接口。

In [2]:
dates = list(pd.date_range('2026-01-01', periods=8, freq='D')) * 2
regimes = ['bull', 'bull', 'bull', 'neutral', 'stress', 'stress', 'stress', 'downtrend'] * 2
daily = pd.DataFrame(
    {
        'date': dates,
        'model': ['ridge_mse'] * 8 + ['gcn_mse'] * 8,
        'regime': regimes,
        'nav': [
            1.00, 1.02, 1.04, 1.03, 1.01, 0.99, 1.00, 1.01,
            1.00, 1.04, 1.08, 1.07, 1.02, 0.96, 0.97, 0.98,
        ],
        'benchmark_nav': [1.0, 1.01, 1.02, 1.02, 1.00, 0.99, 1.00, 1.00] * 2,
        'position': [0.2, 0.3, 0.5, 0.1, 0.4, 0.2, 0.1, 0.2] * 2,
        'turnover': [0.0, 0.1, 0.2, 0.4, 0.3, 0.1, 0.1, 0.2] * 2,
        'vix': [17, 18, 19, 24, 35, 32, 31, 22] * 2,
        'trend': [0.02, 0.03, 0.04, 0.00, -0.05, -0.06, -0.04, -0.02] * 2,
        'drawdown': [0.0, 0.0, 0.0, -0.01, -0.12, -0.14, -0.10, -0.04] * 2,
    }
)
sample_daily_path = CARD_DIR / 'sample_daily_backtest.csv'
daily.to_csv(sample_daily_path, index=False)
daily.head()


,date,model,regime,nav,benchmark_nav,position,turnover,vix,trend,drawdown
0,2026-01-01,ridge_mse,bull,1.00,1.00,0.2,0.0,17,0.02,0.00
1,2026-01-02,ridge_mse,bull,1.02,1.01,0.3,0.1,18,0.03,0.00
2,2026-01-03,ridge_mse,bull,1.04,1.02,0.5,0.2,19,0.04,0.00
3,2026-01-04,ridge_mse,neutral,1.03,1.02,0.1,0.4,24,0.00,-0.01
4,2026-01-05,ridge_mse,stress,1.01,1.00,0.4,0.3,35,-0.05,-0.12


## 框架输出

`build_model_regime_window_diagnostics` 不替代明细表；它从 `model_effect_deltas` 压缩出每个模型一行的稳定性画像，供 dashboard、schedule 和 review 快速定位。

In [3]:
windows = {
    'early': ('2026-01-01', '2026-01-04'),
    'stress_window': ('2026-01-05', '2026-01-07'),
}
analysis = analyze_model_effects_by_regime_and_window(
    daily,
    windows=windows,
    baseline_model='ridge_mse',
)
diagnostics = build_model_regime_window_diagnostics(analysis)
diagnostics


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


,schema_version,model,baseline_model,evaluated_segment_count,positive_segment_count,negative_segment_count,weakness_segment_count,strength_segment_count,risk_segment_count,window_regime_negative_count,...,best_stability_score_delta,worst_segment_type,worst_segment,worst_window,worst_regime,worst_relative_return_pct_delta,worst_stability_score_delta,worst_max_drawdown_pct_delta,diagnostic_signal,review_note
0,analysis.model_regime_window_diagnostic.v1,ridge_mse,ridge_mse,10,0,0,0,0,0,0,...,0.000000,all,all,,,0.000000,0.000000,0.000000,baseline,ridge_mse 是当前分析基准；其他模型的 regime/window 差异均相对它计算。
1,analysis.model_regime_window_diagnostic.v1,gcn_mse,ridge_mse,10,3,4,4,3,4,1,...,-7933.253933,regime,stress,,stress,-3.911862,-68.271389,3.902155,stress_fragile,gcn_mse 在压力或下跌分段弱于基准 ridge_mse，最弱分段为 stress；晋升...


In [4]:
report = build_regime_window_analysis_report(
    artifact_id='analysis_regime_window_diagnostics_v1',
    effect_analysis=analysis,
    model_comparison=compare_models(daily),
    regime_metrics=compute_regime_metrics(daily),
    window_metrics=compute_window_metrics(daily, windows=windows),
    summary={
        'status': 'analysis_regime_window_diagnostics_completed',
        'note': '中文 review 入口：区间/regime 模型效果差异由 src/analysis 统一生成，skill 只提供发散假设。',
    },
)
dataset = DatasetSnapshot(
    name='analysis_regime_window_diagnostics_unit',
    universe=('SOXL', 'SOXX'),
    start='2026-01-01',
    end='2026-01-08',
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=(str(sample_daily_path.relative_to(REPO_ROOT)),),
)
export = write_analysis_report(report, REPORT_DIR, dataset=dataset)
notebook_summary = {
    'schema_version': 'research.notebook_summary.v1',
    'status': 'completed',
    'framework_surface': 'src/analysis',
    'artifact_id': 'analysis_regime_window_diagnostics_v1',
    'diagnostic_count': int(len(diagnostics)),
    'diagnostic_signals': sorted(diagnostics['diagnostic_signal'].astype(str).unique().tolist()),
    'analysis_report_dir': str(CARD_DIR),
    'manifest_path': str(CARD_DIR / 'analysis_manifest.json'),
    'artifact_paths': export['paths'],
    'skill_boundary': 'skill 只用于抽象分析思路和候选 regime，不保存项目状态、不作为运行入口。',
}
(CARD_DIR / 'notebook_summary.json').write_text(
    json.dumps(notebook_summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
notebook_summary


{'schema_version': 'research.notebook_summary.v1',
 'status': 'completed',
 'framework_surface': 'src/analysis',
 'artifact_id': 'analysis_regime_window_diagnostics_v1',
 'diagnostic_count': 2,
 'diagnostic_signals': ['baseline', 'stress_fragile'],
 'analysis_report_dir': '/Users/rziiiii/dev/workspace/research/analysis_regime_window_diagnostics_v1',
 'manifest_path': '/Users/rziiiii/dev/workspace/research/analysis_regime_window_diagnostics_v1/analysis_manifest.json',
 'artifact_paths': {'summary': 'analysis_summary.json',
  'model_comparison': 'model_comparison.csv',
  'regime_metrics': 'regime_metrics.csv',
  'window_metrics': 'window_metrics.csv',
  'window_regime_metrics': 'window_regime_metrics.csv',
  'qlib_return_frame': 'qlib_return_frame.csv',
  'qlib_risk_metrics': 'qlib_risk_metrics.csv',
  'segment_metrics': 'segment_metrics.csv',
  'segment_effect_matrix': 'segment_effect_matrix.csv',
  'model_effect_deltas': 'model_effect_deltas.csv',
  'model_regime_window_diagnostics': '

## 中文结论

- 做了什么：验证 `src/analysis` 可以从 daily backtest/qlib 标准化输入生成 `model_regime_window_diagnostics.csv`，并通过 `analysis_report` 和 `analysis_manifest.json` 暴露给 dashboard/schedule。
- 核心指标：本 notebook 生成 2 个模型画像；`ridge_mse` 为基准，`gcn_mse` 触发 `stress_fragile`，最弱分段集中在 stress/window_regime。
- regime 差异：同一模型可以在 bull 分段有优势，但在 stress/downtrend 分段弱于基准，所以晋升不能只看全区间收益。
- 是否晋升：框架能力可以晋升到稳定 `src/analysis`；样例模型不做生产晋升，只作为功能验证。
- 风险：diagnostics 是二级摘要，不能替代 `model_effect_deltas`、`failure_modes` 和 qlib 风险表；阈值变化需要继续通过测试和 notebook 留证据。
- 下一步：把更多候选 regime 设计放在 skill/研究 notebook 中发散，验证通过后再沉到 `src/analysis` 的显式接口。